In [1]:
import jax
import jax.numpy as jnp

In [2]:
global_list = []

def log2(x):
    global_list.append(x)
    ln_x = jnp.log(x)
    ln_2 = jnp.log(2.0)
    return ln_x / ln_2

print(jax.make_jaxpr(log2)(3.0))

{ lambda ; a:f32[]. let
    b:f32[] = log a
    c:f32[] = log 2.0:f32[]
    d:f32[] = div b c
  in (d,) }


No mention of the global - which is expected, since JAX transformations are designed to understand side-effect-free code.

In [3]:
def log2_with_print(x):
    print("printed x:", x)
    ln_x = jnp.log(x)
    ln_2 = jnp.log(2.0)
    return ln_x / ln_2

print(jax.make_jaxpr(log2_with_print)(3.))

printed x: JitTracer(~float32[])
{ lambda ; a:f32[]. let
    b:f32[] = log a
    c:f32[] = log 2.0:f32[]
    d:f32[] = div b c
  in (d,) }


The print function is impure, so it occurs during the tracing, but not in the jaxpr.

In [4]:
def log2_if_rank_2(x):
    if x.ndim == 2:
        ln_x = jnp.log(x)
        ln_2 = jnp.log(2.)
        return ln_x / ln_2
    else:
        return x

print(jax.make_jaxpr(log2_if_rank_2)(jax.numpy.array([1,2,3])))

{ lambda ; a:i32[3]. let  in (a,) }


## JIT compiling a function

In [9]:
def selu(x, alpha=1.67, lambda_=1.05):
    return lambda_ * jnp.where(x > 0, x, alpha * jnp.exp(x) - alpha)

x = jnp.arange(1000000)
%timeit selu(x).block_until_ready()

365 μs ± 15.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [11]:
selu_jit = jax.jit(selu)

# pre-compile the function before timing
selu_jit(x).block_until_ready()

%timeit selu_jit(x).block_until_ready()

157 μs ± 290 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## JAX Conditionals

In [14]:
@jax.jit
def loop_body(prev_i):
    return prev_i + 1

def g_inner_jitted(x, n):
    i = 0
    while i < n:
        i = loop_body(i)
    return x + i

g_inner_jitted(10, 20)

Array(30, dtype=int32, weak_type=True)

In [17]:
@jax.jit(static_argnames=['n'])
def g_with_static(x, n):
    i=0
    while i < n:
        i = i + 1
    return x + i

g_with_static(10, 20)

Array(30, dtype=int32, weak_type=True)

In [18]:
from functools import partial

def unjitted_loop_body(prev_i):
    return prev_i + 1

def g_inner_jitted_partial(x, n):
    i = 0
    while i < n:
        # each iteration, partial returns a function with a different hash
        i = jax.jit(partial(unjitted_loop_body))(i)
    return x + i

def g_inner_jitted_lambda(x, n):
    i = 0
    while i < n:
        # each iteration, lambda returns a function with a different hash
        i = jax.jit(lambda x: unjitted_loop_body(x))(i)
    return x + i

def g_inner_jitted_normal(x, n):
    i = 0
    while i < n:
        # hash for the compiled function is consistent
        i = jax.jit(unjitted_loop_body)(i)
    return x + i


print("jit called in a loop with partials:")
%timeit g_inner_jitted_partial(10, 20).block_until_ready()

print("jit called in a loop with lambdas:")
%timeit g_inner_jitted_lambda(10, 20).block_until_ready()

print("jit called in a loop with caching:")
%timeit g_inner_jitted_normal(10, 20).block_until_ready()

jit called in a loop with partials:
158 ms ± 6.47 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
jit called in a loop with lambdas:
152 ms ± 5.73 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
jit called in a loop with caching:
412 μs ± 1.07 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
